In [ ]:
def create_F1_population(
    ancestor_poAncestry_1: List['Individual'],
    ancestor_poAncestry_2: List['Individual'],
    recomb_event_probabilities: dict,
    recomb_probabilities: List[float]
) -> List['Individual']:
    """
    Generates the first filial (F1) hybrid population by crossing paired individuals
    from two distinct ancestral populations.

    Each F1 individual is created by simulating gamete formation (meiosis with recombination)
    from one individual of `ancestor_poAncestry_1` and one from `ancestor_poAncestry_2`,
    and then combining these gametes. The genetic and recombination data for each
    F1 offspring is immediately recorded.

    Args:
        ancestor_poAncestry_1 (List[Individual]): The first ancestral population (e.g., all 'A2' homozygotes).
        ancestor_poAncestry_2 (List[Individual]): The second ancestral population (e.g., all 'A0' homozygotes).
        recomb_event_probabilities (dict): Probability distribution for the number of recombination events per chromosome.
        recomb_probabilities (List[float]): Position-dependent probabilities for recombination along chromosomes.

    Returns:
        List[Individual]: A list containing all the newly created F1 hybrid individuals.

    Raises:
        ValueError: If the input ancestral populations are not of the same size,
                    as paired crosses cannot be performed reliably.
    """
    # Ensure that both ancestral populations have an equal number of individuals for paired crossing.
    if len(ancestor_poAncestry_1) != len(ancestor_poAncestry_2):
        raise ValueError("Error: Ancestor populations must be the same size to create F1 population via paired crosses.")

    f1_population = [] # Initialize an empty list to collect the F1 individuals.

    # Iterate through the ancestral populations, pairing individuals by their index
    # to perform crosses. Each pair produces one F1 offspring.
    for i in range(len(ancestor_poAncestry_1)):
        ancestor_A = ancestor_poAncestry_1[i] # Select an individual from the first ancestral population.
        ancestor_B = ancestor_poAncestry_2[i] # Select a corresponding individual from the second ancestral population.

        # Create a new 'Individual' instance to represent the F1 offspring.
        # The F1 offspring will inherit the same chromosome and locus structure as its parents.
        # The `generation_label='F1'` is passed to the Individual constructor (assuming your updated Individual class).
        progeny = Individual(ancestor_A.num_chromosomes, ancestor_A.num_loci_per_chromosome, generation_label='F1')
        # Note: The original code had `offspring.diploid_chromosome_pairs = []` here.
        # This line is redundant if the Individual.__init__ correctly initializes it as an empty list,
        # but does no harm.

        # For each homologous chromosome pair in the parents, simulate meiosis and combine gametes.
        for chr_idx in range(ancestor_A.num_chromosomes):
            # Get the specific diploid chromosome pair from Ancestor A.
            chr_A_pair = ancestor_A.diploid_chromosome_pairs[chr_idx]
            # Get the specific diploid chromosome pair from Ancestor B.
            chr_B_pair = ancestor_B.diploid_chromosome_pairs[chr_idx]

            # Simulate meiosis for Ancestor A's chromosome pair to get one haploid chromatid (gamete).
            haploid_A = meiosis_with_recombination(chr_A_pair, recomb_event_probabilities, recomb_probabilities)
            # Simulate meiosis for Ancestor B's chromosome pair to get one haploid chromatid (gamete).
            haploid_B = meiosis_with_recombination(chr_B_pair, recomb_event_probabilities, recomb_probabilities)

            # Combine the two haploid chromatids (gametes) to form a new diploid pair for the F1 offspring.
            progeny.diploid_chromosome_pairs.append(DiploidChromosomePair(haploid_A, haploid_B))

        # After the F1 offspring individual is fully constructed, record its genetic and recombination data.
        # This adds the individual's full genotype and chromatid block data to global lists for analysis.
        record_individual_genome(progeny, 'F1')
        record_chromatid_recombination(progeny, 'F1')

        f1_population.append(progeny) # Add the newly created F1 individual to the F1 population list.

    return f1_population # Return the complete list of F1 individuals.